# Confronto spaziale MTGFlow–STGAN e forecasting t+1/t+6

Analisi esclusivamente post-hoc. Ogni località PVGIS rimane un punto distinto: il vicinato non viene ridotto a un singolo pixel. Accanto alle mappe regionali vengono prodotti lo zoom sulla località di riferimento più gli 8 vicini geografici e una serie temporale aggregata del vicinato.

In [ ]:
from pathlib import Path
import json, os, sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from physiq_pv.reporting import mtgflow_spatiotemporal as mtg_spatial
from physiq_pv.reporting.detector_spatial_comparison import (
    aggregate_detector_locations, aggregate_detector_timeline,
    aggregate_forecast_locations, aggregate_forecast_timeline,
    load_detector_event_rows, load_forecast_event_rows,
    select_reference_neighbourhood,
)

MTGFLOW_SEED_DIR = Path(os.environ.get('MTGFLOW_SEED_DIR', ROOT / 'outputs/pvgis_mtgflow/downstream_dense/seed_15')).resolve()
STGAN_SEED_DIR = Path(os.environ.get('STGAN_SEED_DIR', ROOT / 'outputs/pvgis_stgan/paper_reference/seed_20')).resolve()
SDE_RUN = 'pvgis_stgnn_paper_faithful_gaussian_detector_mtgflow_ep60_h1-2-3-4-5-6_direct_seed1'
SDE_PREDICTIONS = Path(os.environ.get('SDE_MULTIHORIZON_PREDICTIONS', ROOT / 'outputs' / SDE_RUN / 'predictions.csv')).resolve()
PVGIS_2019 = Path(os.environ.get('PVGIS_2019_PATH', '/data/SentinelPV/pvgis_data/data/pvgis_summed_irradiance/piedmont_pvgis_2019.nc')).resolve()
TRAINING_STATS = Path(os.environ.get('MTGFLOW_TRAINING_STATS_CSV', ROOT / 'outputs/mtgflow_threshold_sensitivity/t_plus_6/training_iqr_by_location.csv')).resolve()
OUT_DIR = Path(os.environ.get('SPATIAL_COMPARISON_OUT_DIR', ROOT / 'outputs/anomaly_spatial_comparison')).resolve()
FIGURE_DIR = OUT_DIR / 'figures'; FIGURE_DIR.mkdir(parents=True, exist_ok=True)
HORIZONS = (1, 6)
N_NEIGHBOURS = 8
REFERENCE_LOCATION = os.environ.get('PVGIS_REFERENCE_LOCATION') or None
DAYTIME_THRESHOLD_WM2 = 10.0
EVENTS = {
    'april_dust_23_26': ('2019-04-23', '2019-04-24', '2019-04-25', '2019-04-26'),
    'june_extreme_28_29': ('2019-06-28', '2019-06-29'),
    'july_regional_peak': ('2019-07-02',),
}
print('MTGFlow:', MTGFLOW_SEED_DIR)
print('STGAN  :', STGAN_SEED_DIR)
print('SDE     :', SDE_PREDICTIONS)
print('Output  :', OUT_DIR)

## 1. Coordinate, threshold storiche e vicinato indipendente dagli score

In [ ]:
MTGFLOW_SCORES = MTGFLOW_SEED_DIR / 'anomaly_scores.csv'
STGAN_SCORES = STGAN_SEED_DIR / 'anomaly_scores.csv'
required = (MTGFLOW_SCORES, STGAN_SCORES, SDE_PREDICTIONS, PVGIS_2019)
missing = [path for path in required if not path.is_file()]
if missing: raise FileNotFoundError('File mancanti:\n' + '\n'.join(map(str, missing)))
locations, pvgis_times, poa = mtg_spatial.load_pvgis_spatial_context(PVGIS_2019)
saved = mtg_spatial.read_saved_thresholds(MTGFLOW_SCORES)
threshold_table = mtg_spatial.build_threshold_table(
    saved, cached_statistics=TRAINING_STATS,
    per_site_training_paths=sorted(MTGFLOW_SEED_DIR.glob('*/train_scores.csv')),
    aggregate_training_path=MTGFLOW_SEED_DIR / 'train_anomaly_scores.csv',
)
neighbourhood = select_reference_neighbourhood(
    locations, n_neighbours=N_NEIGHBOURS, reference_location=REFERENCE_LOCATION
)
neighbourhood.to_csv(OUT_DIR / 'reference_neighbourhood.csv', index=False)
print('Località di riferimento:', neighbourhood.iloc[0]['location'])
display(neighbourhood)

## 2. Stessi eventi e stesso filtro diurno per MTGFlow e STGAN

In [ ]:
mtg_rows = load_detector_event_rows(
    MTGFLOW_SCORES, detector='mtgflow', events=EVENTS,
    threshold_table=threshold_table, pvgis_locations=locations,
    pvgis_times=pvgis_times, poa_by_location_time=poa,
    daytime_threshold_wm2=DAYTIME_THRESHOLD_WM2,
)
stgan_rows = load_detector_event_rows(
    STGAN_SCORES, detector='stgan', events=EVENTS,
    pvgis_locations=locations, pvgis_times=pvgis_times,
    poa_by_location_time=poa, daytime_threshold_wm2=DAYTIME_THRESHOLD_WM2,
)
detector_rows = pd.concat([mtg_rows, stgan_rows], ignore_index=True)
detector_locations = aggregate_detector_locations(detector_rows)
regional_detector_timeline = aggregate_detector_timeline(detector_rows)
neighbour_ids = neighbourhood['location'].astype(str).tolist()
neighbour_detector_timeline = aggregate_detector_timeline(detector_rows, locations=neighbour_ids)
detector_locations.to_csv(OUT_DIR / 'event_detector_metrics_by_location.csv', index=False)
regional_detector_timeline.to_csv(OUT_DIR / 'regional_detector_timeline.csv', index=False)
neighbour_detector_timeline.to_csv(OUT_DIR / 'neighbourhood_detector_timeline.csv', index=False)
display(detector_locations.groupby(['detector', 'event'])[['n_anomalies', 'n_observations']].sum())

## 3. Errori sulle stesse coordinate target, separati a t+1 e t+6

In [ ]:
forecast_rows = load_forecast_event_rows(
    SDE_PREDICTIONS, events=EVENTS, horizons=HORIZONS,
    daytime_threshold_wm2=DAYTIME_THRESHOLD_WM2,
)
forecast_locations = aggregate_forecast_locations(forecast_rows)
regional_forecast_timeline = aggregate_forecast_timeline(forecast_rows)
neighbour_forecast_timeline = aggregate_forecast_timeline(forecast_rows, locations=neighbour_ids)
forecast_locations.to_csv(OUT_DIR / 'event_forecast_metrics_by_location.csv', index=False)
regional_forecast_timeline.to_csv(OUT_DIR / 'regional_forecast_timeline.csv', index=False)
neighbour_forecast_timeline.to_csv(OUT_DIR / 'neighbourhood_forecast_timeline.csv', index=False)
display(forecast_locations.groupby(['event', 'horizon_hours'])[['n_forecasts', 'sum_abs_error', 'sum_squared_error']].sum())

## 4. Mappe regionali e zoom KNN

L'intensità MTGFlow è il superamento positivo della threshold in IQR storici; l'intensità STGAN è il percentile globale. Le due scale sono dichiaratamente diverse e non vengono confrontate numericamente.

In [ ]:
def _panel(axis, frame, value, title, cmap, vmin=None, vmax=None):
    image = axis.scatter(frame['longitude'], frame['latitude'], c=frame[value], cmap=cmap, vmin=vmin, vmax=vmax, marker='s', s=26, linewidths=0)
    axis.set(title=title, xlabel='Longitudine', ylabel='Latitudine')
    axis.set_aspect(1.0 / np.cos(np.deg2rad(frame['latitude'].mean())))
    axis.grid(alpha=0.12); return image

def build_comparison_map(event, selected_locations=None, suffix='regional'):
    coords = locations.copy()
    if selected_locations is not None: coords = coords[coords['location'].astype(str).isin(set(map(str, selected_locations)))]
    det = detector_locations[detector_locations['event'].eq(event)].merge(coords, on='location', how='inner')
    pred = forecast_locations[forecast_locations['event'].eq(event)].merge(coords, on='location', how='inner')
    mtg = det[det['detector'].eq('mtgflow')]; stg = det[det['detector'].eq('stgan')]
    h1 = pred[pred['horizon_hours'].eq(1)]; h6 = pred[pred['horizon_hours'].eq(6)]
    fig, axes = plt.subplots(2, 3, figsize=(18, 11))
    panels = [
        (mtg, 'intensity_max', 'MTGFlow: max superamento [IQR]', 'magma', 0, None),
        (stg, 'intensity_max', 'STGAN: max percentile globale', 'magma', 0, 1),
        (mtg, 'anomaly_fraction', 'MTGFlow: frazione anomala', 'Reds', 0, 1),
        (stg, 'anomaly_fraction', 'STGAN: frazione anomala', 'Reds', 0, 1),
        (h1, 'rmse', 'SDE-Net RMSE t+1', 'viridis', 0, None),
        (h6, 'rmse', 'SDE-Net RMSE t+6', 'viridis', 0, None),
    ]
    for axis, args in zip(axes.flat, panels):
        image = _panel(axis, *args); fig.colorbar(image, ax=axis, shrink=0.78)
    fig.suptitle(f'{event} — confronto spaziale {suffix}', y=1.01); fig.tight_layout()
    path = FIGURE_DIR / f'{event}_spatial_comparison_{suffix}.png'
    fig.savefig(path, dpi=180, bbox_inches='tight'); plt.show(); return path

map_paths = []
for event in EVENTS:
    map_paths.append(build_comparison_map(event))
    map_paths.append(build_comparison_map(event, neighbour_ids, 'reference_knn'))
print('Mappe create:', len(map_paths))

## 5. Aggregato del vicinato: non sostituisce i pixel della mappa

In [ ]:
timeline_paths = []
for event in EVENTS:
    det = neighbour_detector_timeline[neighbour_detector_timeline['event'].eq(event)]
    pred = neighbour_forecast_timeline[neighbour_forecast_timeline['event'].eq(event)]
    fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
    for detector, frame in det.groupby('detector', observed=True):
        axes[0].plot(frame['timestamp'], 100 * frame['anomaly_fraction'], label=detector)
    for horizon, frame in pred.groupby('horizon_hours', observed=True):
        axes[1].plot(frame['timestamp'], frame['rmse'], label=f't+{horizon}')
    axes[0].set(ylabel='Località anomale [%]', title='Vicinato: frazione anomala'); axes[0].legend(); axes[0].grid(alpha=.25)
    axes[1].set(ylabel='RMSE [W]', xlabel='Timestamp target', title='Vicinato: errore SDE-Net'); axes[1].legend(); axes[1].grid(alpha=.25)
    fig.suptitle(event); fig.tight_layout()
    path = FIGURE_DIR / f'{event}_reference_neighbourhood_timeline.png'
    fig.savefig(path, dpi=180, bbox_inches='tight'); plt.show(); timeline_paths.append(path)
print('Timeline create:', len(timeline_paths))

In [ ]:
metadata = {
    'post_processing_only': True, 'training_rerun': False,
    'events': {key: list(value) for key, value in EVENTS.items()},
    'horizons_hours': list(HORIZONS),
    'reference_location': str(neighbourhood.iloc[0]['location']),
    'neighbourhood_rule': f'reference_plus_{N_NEIGHBOURS}_geographical_nearest',
    'spatial_pixels_aggregated': False,
    'temporal_aggregation': ['max_score', 'anomaly_fraction', 'mae', 'rmse'],
    'mtgflow_intensity': 'max((score-threshold)/training_iqr, 0)',
    'stgan_intensity': 'global_score_percentile/100',
    'daytime_threshold_wm2': DAYTIME_THRESHOLD_WM2,
}
(OUT_DIR / 'analysis_metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
print('Output:', OUT_DIR)